# Undervalued hitters

Pre-registered in `docs/model_undervalued.md`. 
Does skill predict next-season wOBA beyond regression to the mean?

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not find the project root (folder containing src/).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import clean_woba, plate_appearances
from src.models import batter_score as bs

SEASONS = [2021, 2022, 2023, 2024]
MIN_PA_NEXT = 200
SHRINK_K = [100, 300, 600]


def season_table(season):
    """Batter profiles for one season, plus PA and the league wOBA."""
    df = load_all_snapshots(seasons=[season])
    prof = bs.build_profiles(df)
    pa = clean_woba(plate_appearances(df))
    num = pd.to_numeric(pa["woba_value"], errors="coerce")
    den = pd.to_numeric(pa["woba_denom"], errors="coerce")
    league = float(num.sum() / den.sum())
    prof = prof.join(den.groupby(pa["batter"]).sum().rename("pa"))
    return prof, league


tables = {}
for s in SEASONS:
    tables[s] = season_table(s)
    print(f"{s}: {len(tables[s][0])} batters, league wOBA {tables[s][1]:.3f}")

rows = []
for t in SEASONS[:-1]:
    prof_t, league_t = tables[t]
    prof_n, _ = tables[t + 1]

    model = bs.fit(prof_t)  # season t only
    q = bs.qualified(prof_t)
    q = q.assign(score=model.score(q))

    nxt = prof_n.loc[prof_n["pa"] >= MIN_PA_NEXT, "woba"].rename("woba_next")
    d = q.join(nxt, how="inner")
    y = d["woba_next"].astype("float64")
    w_t = d["woba"].astype("float64")

    preds = {"last_season": w_t}
    for k in SHRINK_K:
        weight = d["pa"] / (d["pa"] + k)
        preds[f"regressed_k{k}"] = weight * w_t + (1 - weight) * league_t
    preds["skill_score"] = d["score"]

    for name, p in preds.items():
        rows.append({"pair": f"{t}->{t+1}", "n": len(d), "method": name,
                     "mae": float((p - y).abs().mean())})

r = pd.DataFrame(rows)
table = r.pivot(index="method", columns="pair", values="mae")
print()
print(table.round(4))
print()
print("n per pair:", r.groupby("pair")["n"].first().to_dict())

baselines = [m for m in table.index if m != "skill_score"]
best = table.loc[baselines].min()
gain = (best - table.loc["skill_score"]) / best
print()
print("Skill score vs the BEST baseline in each pair (positive = skill wins):")
for pair, g in gain.items():
    print(f"  {pair}: {g*100:+.2f}%  (best baseline: {table.loc[baselines, pair].idxmin()})")
print()
print("Pre-registered verdict:", "PASS" if (gain > 0).all() else "FAIL")


2021: 438 batters, league wOBA 0.318
2022: 435 batters, league wOBA 0.310
2023: 433 batters, league wOBA 0.321
2024: 425 batters, league wOBA 0.312

pair            2021->2022  2022->2023  2023->2024
method                                            
last_season         0.0339      0.0276      0.0320
regressed_k100      0.0306      0.0256      0.0302
regressed_k300      0.0276      0.0247      0.0288
regressed_k600      0.0262      0.0252      0.0282
skill_score         0.0290      0.0255      0.0279

n per pair: {'2021->2022': 246, '2022->2023': 253, '2023->2024': 267}

Skill score vs the BEST baseline in each pair (positive = skill wins):
  2021->2022: -10.65%  (best baseline: regressed_k600)
  2022->2023: -3.27%  (best baseline: regressed_k300)
  2023->2024: +0.77%  (best baseline: regressed_k600)

Pre-registered verdict: FAIL


In [2]:
df = load_all_snapshots(seasons=[2023])
pa = plate_appearances(df)
den = pd.to_numeric(pa["woba_denom"], errors="coerce")
num = pd.to_numeric(pa["woba_value"], errors="coerce")

print("PA rows:", len(pa))
print("woba_denom sum:", den.sum(), "| NaN denom:", den.isna().sum(), "| zero denom:", (den == 0).sum())
print()
if "game_type" in pa.columns:
    print("game_type:", pa["game_type"].value_counts().to_dict())
    print()
print("Events with missing or zero denominator:")
print(pa.loc[den.isna() | (den == 0), "events"].value_counts().head(15))
print()
print("Strikeouts:", (pa["events"] == "strikeout").sum(),
      "| strikeout denom sum:", den[pa["events"] == "strikeout"].sum())

PA rows: 184376
woba_denom sum: 182706 | NaN denom: 973 | zero denom: 697

game_type: {'R': 184376}

Events with missing or zero denominator:
events
intent_walk                  474
sac_bunt                     429
truncated_pa                 272
field_out                    237
strikeout                     84
single                        73
walk                          55
force_out                     16
grounded_into_double_play     11
fielders_choice_out            4
double                         4
field_error                    4
fielders_choice                3
home_run                       2
sac_fly                        1
Name: count, dtype: int64

Strikeouts: 41743 | strikeout denom sum: 41659


In [3]:
pa2 = pa.assign(num=num, den=den)
by_event = (pa2.groupby("events")
               .agg(count=("num", "size"),
                    num_sum=("num", "sum"),
                    num_mean=("num", "mean"),
                    den_sum=("den", "sum"))
               .sort_values("num_sum", ascending=False))
print(by_event.round(3).to_string())
print()
print(f"Total numerator: {num.sum():,.0f}")
print(f"Total denominator: {den.sum():,.0f}")
print(f"Our league wOBA: {num.sum() / den.sum():.4f}  (Savant: .316)")
print()
print("Rows with a numerator but a missing or zero denominator:")
mask = (num > 0) & (den.isna() | (den == 0))
print(pa2.loc[mask, "events"].value_counts())
print(f"Their numerator total: {num[mask].sum():,.1f}")

                           count  num_sum  num_mean  den_sum
events                                                      
single                     26031  23427.9       0.9    25958
home_run                    5868  11736.0       2.0     5866
walk                       15345  10741.5       0.7    15290
double                      8228  10285.0      1.25     8224
hit_by_pitch                2112   1478.4       0.7     2112
triple                       712   1139.2       1.6      711
field_error                 1066    959.4       0.9     1062
fielders_choice              377    337.5     0.895      374
intent_walk                  474    189.6       0.4        0
sac_bunt                     429    108.9     0.254        0
catcher_interf                96     67.2       0.7       96
strikeout                  41743     51.8     0.001    41659
sac_fly                     1206      4.5     0.004     1205
grounded_into_double_play   3455      0.0       0.0     3444
force_out               

In [2]:
# Numbers for docs/model_batter_score.md, recomputed after the wOBA fix.
# Reuses the profiles built by the first cell, so run that first.
from scipy.stats import spearmanr, pearsonr

YEARS = [2021, 2022, 2023, 2024]
models = {y: bs.fit(tables[y][0]) for y in YEARS}
quals = {y: bs.qualified(tables[y][0]) for y in YEARS}

print("Per-season batter weights and R^2")
w = pd.DataFrame({y: models[y].weights for y in YEARS}).T
w["R2"] = [models[y].r_squared for y in YEARS]
print(w.round(4).to_string())
cv = w[bs.SKILL_METRICS].std() / w[bs.SKILL_METRICS].mean().abs() * 100
print("\nCoefficient of variation (%):")
print(cv.round(1).to_string())

print("\nRank correlation (rows: players' season, columns: weights' season)")
rc = pd.DataFrame(index=YEARS, columns=YEARS, dtype="float64")
for s in YEARS:
    own = models[s].score(quals[s])
    for wy in YEARS:
        rc.loc[s, wy] = spearmanr(own, models[wy].score(quals[s])).correlation
print(rc.round(4).to_string())

common = sorted(set.intersection(*[set(quals[y].index) for y in YEARS]))
sc = pd.DataFrame({y: models[2024].score(quals[y].loc[common]) for y in YEARS})
print(f"\n{len(common)} batters qualified in all four seasons, scored with 2024 weights:")
for a, b2 in zip(YEARS[:-1], YEARS[1:]):
    print(f"  {a}->{b2}: pearson {pearsonr(sc[a], sc[b2])[0]:.3f}  "
          f"spearman {spearmanr(sc[a], sc[b2]).correlation:.3f}")

print("\n2024 examples:")
q24 = quals[2024].assign(pred=models[2024].score(quals[2024]))
for name, pid in [("Judge", 592450), ("Stanton", 519317), ("Seager", 608369)]:
    if pid in q24.index:
        r_ = q24.loc[pid]
        print(f"  {name}: predicted {r_['pred']:.3f}  actual {r_['woba']:.3f}  "
              f"gap {r_['pred'] - r_['woba']:+.3f}")
    else:
        print(f"  {name}: id {pid} not among 2024 qualified batters")


Per-season batter weights and R^2
      chase_pct  zone_swing_pct  zone_contact_pct  barrel_pct      R2
2021    -0.0095          0.0077            0.0165      0.0300  0.5032
2022    -0.0031          0.0030            0.0193      0.0288  0.4795
2023    -0.0070          0.0032            0.0159      0.0288  0.4940
2024    -0.0084          0.0068            0.0183      0.0292  0.5197

Coefficient of variation (%):
chase_pct           40.1
zone_swing_pct      46.7
zone_contact_pct     9.1
barrel_pct           1.9

Rank correlation (rows: players' season, columns: weights' season)
        2021    2022    2023    2024
2021  1.0000  0.9681  0.9901  0.9975
2022  0.9671  1.0000  0.9832  0.9801
2023  0.9874  0.9796  1.0000  0.9930
2024  0.9962  0.9751  0.9914  1.0000

145 batters qualified in all four seasons, scored with 2024 weights:
  2021->2022: pearson 0.654  spearman 0.621
  2022->2023: pearson 0.751  spearman 0.705
  2023->2024: pearson 0.720  spearman 0.674

2024 examples:
  Judge: predi